In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
import nltk
import numpy as np
from collections import Counter, defaultdict
from scipy.sparse import coo_matrix

In [ ]:
DOWNLOAD_DIR = "/tmp"
nltk.download("brown", download_dir=DOWNLOAD_DIR)

In [ ]:
from nltk.corpus import brown
tokenized_sentences = brown.sents()
tokenized_sentences = [" ".join(sentence) for sentence in tokenized_sentences]
tokenized_sentences = [sentence.lower() for sentence in tokenized_sentences]

print(tokenized_sentences[0])

In [4]:
from collections import Counter, defaultdict
from scipy.sparse import coo_matrix
import string


def build_windowed_cooccurrence(tokenized_sentences, vocab_size=None, window_size=5, min_count=1):
    """build a symmetric sliding-window co-occurrence matrix for GloVe.

    Args:
        tokenized_sentences: list of strings or list of token lists
        vocab_size: optional max vocabulary size; keeps the most frequent words
        window_size: number of tokens to the left/right to consider
        min_count: ignore co-occurrence pairs below this value

    Returns:
        co_occurrence_matrix: scipy CSR sparse matrix of shape [vocab_size, vocab_size]
        vocab: dict mapping word -> index
    """
    if not tokenized_sentences:
        raise ValueError("tokenized_sentences is empty")

    # count token frequencies first for optional vocab trimming.
    token_counts = Counter()
    for sentence in tokenized_sentences:
        if isinstance(sentence, str):
            tokens = sentence.split()
        else:
            tokens = sentence
        tokens = [t for t in tokens if not all(c in string.punctuation for c in t)]
        token_counts.update(tokens)

    if vocab_size is not None:
        vocab_words = [w for w, f in token_counts.most_common(vocab_size) if f >= min_count]
    else:
        vocab_words = list(token_counts.keys())

    vocab = {word: idx for idx, word in enumerate(vocab_words)}

    cooc = defaultdict(float)
    for sentence in tokenized_sentences:
        if isinstance(sentence, str):
            tokens = sentence.split()
        else:
            tokens = sentence

        for i, center in enumerate(tokens):
            if center not in vocab:
                continue

            left = max(0, i - window_size)
            right = min(len(tokens), i + window_size + 1)

            for j in range(left, right):
                if i == j:
                    continue
                context = tokens[j]
                if context not in vocab:
                    continue

                dist = abs(i - j)
                weight = 1.0 / dist
                cooc[(vocab[center], vocab[context])] += weight

    rows, cols, data = [], [], []
    for (i, j), value in cooc.items():
        rows.append(i)
        cols.append(j)
        data.append(value)

    co_occurrence_matrix = coo_matrix((data, (rows, cols)), shape=(len(vocab), len(vocab)))
    co_occurrence_matrix.setdiag(0)
    co_occurrence_matrix.eliminate_zeros()
    co_occurrence_matrix = co_occurrence_matrix.tocsr()
    
    return co_occurrence_matrix, vocab

In [5]:
# build GloVe-style sliding-window co-occurrence counts instead of sentence-level bag-of-words counts.
VOCAB_SIZE = 10000 
co_occurrence_matrix, vocab = build_windowed_cooccurrence(
    tokenized_sentences,
    vocab_size=VOCAB_SIZE,
    window_size=5,
    min_count=5
)
co_occ_matrix = co_occurrence_matrix.tocoo()
print(f"co_occurrence shape: {co_occurrence_matrix.shape}")
print(f"nonzero entries: {co_occ_matrix.nnz}")
print(f"vocab size: {len(vocab)}")

co_occurrence shape: (10000, 10000)
nonzero entries: 1905836
vocab size: 10000


In [6]:
from torch.utils.data import TensorDataset

# create batch dataset
X = []
y = []

for i, j, val in zip(co_occ_matrix.row, co_occ_matrix.col, co_occ_matrix.data):
    X.append([i, j])
    y.append(val)

X_tensor = torch.tensor(X, dtype=torch.long)
y_tensor = torch.tensor(y, dtype=torch.float32)

dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

In [7]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [8]:
class GloVe(nn.Module):
    def __init__(self, vocab_size, embedding_size):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding_size = embedding_size

        self.word_embeddings = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_size)
        self.context_embeddings = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_size)
        self.word_biases = nn.Embedding(num_embeddings=vocab_size, embedding_dim=1)
        self.context_biases = nn.Embedding(num_embeddings=vocab_size, embedding_dim=1)

        init_range = 0.5 / embedding_size
        nn.init.uniform_(self.word_embeddings.weight, -init_range, init_range)
        nn.init.uniform_(self.context_embeddings.weight, -init_range, init_range)
        nn.init.zeros_(self.word_biases.weight)
        nn.init.zeros_(self.context_biases.weight)

    def forward(self, ids):
        word_ids = ids[:,0]
        context_ids = ids[:,1]
        w = self.word_embeddings(word_ids)
        c = self.context_embeddings(context_ids)
        dot = (w * c).sum(dim=1)
        bw = self.word_biases(word_ids).squeeze()
        bc = self.context_biases(context_ids).squeeze()
        return dot + bw + bc

# initialize model with actual vocab size to ensure embedding indices align with `vocab` mapping
model = GloVe(len(vocab), 100).to(device)
print(model)

GloVe(
  (word_embeddings): Embedding(10000, 100)
  (context_embeddings): Embedding(10000, 100)
  (word_biases): Embedding(10000, 1)
  (context_biases): Embedding(10000, 1)
)


In [9]:
class GloVeLoss(nn.Module):
    def __init__(self, co_occ_matrix, x_max=100, alpha=0.75):
        super().__init__()
        self.co_occ_matrix = co_occ_matrix
        self.x_max = x_max
        self.alpha = alpha
    
    def get_weight(self, x):
        # performs elementwise
        condition = x < self.x_max
        true_values = (x / self.x_max)**self.alpha
        false_values = 1.0
        return torch.where(condition, true_values, false_values)

    def forward(self, prediction, count):
        # prediction is going to be w^Tw+bi+b (computed in training loop)
        # returns a scalar output, not a tensor (regression)
        x = count.float()
        weight = self.get_weight(x)
        safe_count = torch.clip(x, min=1e-12, max=None)
        loss = weight*(prediction - torch.log(safe_count))**2
        return torch.mean(loss, dim=0)

In [10]:
percentiles = np.percentile(co_occ_matrix.data, [90, 95, 99])
print(percentiles, (percentiles[0] + percentiles[1]) / 2)
X_MAX = (percentiles[0] + percentiles[1]) / 2
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.999))
loss_function = GloVeLoss(co_occ_matrix, x_max=X_MAX)
print("optimizer set to Adam lr=1e-3")

[ 2.2         4.26666667 17.9       ] 3.2333333333333334
optimizer set to Adam lr=1e-3


In [11]:
def train(model, dataloader, loss_fn, optimizer, device):
    epoch_loss = 0.0
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device).float()
        pred = model(X)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

        if batch % 100 == 0:
            print(f"loss: {loss.item():>7f}  [{batch:>5d}/{len(dataloader):>5d}]")
    print(f"Average Loss: {epoch_loss / len(dataloader):>7f}")

In [ ]:
EPOCHS = 10 

for i in range(EPOCHS):
    print(f"Epoch {i}")
    train(model, dataloader, loss_function, optimizer, device)

In [ ]:
trained_weights = model.word_embeddings.weight.data
trained_context_weights = model.context_embeddings.weight.data

# Convert to numpy for external tools (e.g., scikit-learn or visualization)
weights_numpy = trained_weights.cpu().numpy()
context_weights_numpy = trained_context_weights.cpu().numpy()
print(weights_numpy[5])

In [ ]:
def get_closest_words_from_index(vocab_dict, weights, context_weights, index, n):
    # accept index==0 and None explicitly
    if index is None:
        return []

    # combine word and context embeddings
    word_vector = weights[index] + context_weights[index]
    word_vector = word_vector.astype(float)
    w_norm = np.linalg.norm(word_vector) + 1e-12

    similarities = []
    for word, i in vocab_dict.items():
        current_vector = weights[i] + context_weights[i]
        c_norm = np.linalg.norm(current_vector) + 1e-12
        cosine = np.dot(word_vector, current_vector) / (w_norm * c_norm)
        similarities.append({"word": word, "similarity": float(cosine)})

    similarities_sorted = sorted(similarities, key=lambda w: w["similarity"], reverse=True)
    return [w["word"] for w in similarities_sorted[:n]]

In [18]:
# retrieve index of word from the actual co-occurrence vocab
print(weights_numpy.shape)
word_index = vocab.get("man")
print(word_index)
print(weights_numpy[word_index].shape)

# use the vocab mapping (word -> index) when querying nearest neighbors
similarities = get_closest_words_from_index(vocab, weights_numpy, context_weights_numpy, word_index, 50)
print(similarities)

(10000, 100)
80
(100,)
['man', 'woman', 'young', 'old', 'who', 'boy', 'wife', 'was', 'like', 'one', 'a', 'with', 'another', 'as', 'little', 'god', 'big', 'the', 'good', 'could', 'his', 'himself', 'had', 'girl', 'he', 'but', 'said', 'my', 'if', 'every', 'friend', 'hand', 'she', 'mother', 'her', 'white', 'car', 'once', 'this', 'or', 'only', 'me', 'son', 'face', 'of', 'to', 'dead', 'very', 'told', 'name']


In [19]:
# save the learned weights and vocab sparse matrix
np.savez("glove_model.npz", weights=weights_numpy, context=context_weights_numpy, vocab=vocab)